# Phase 4 — LLM arm

Runs `smart_spatial_system`'s `LLMQuerySpecGenerator` on the *same*
natural-language question the rule-based arm got (phase 3), `N` times,
and saves every run's raw `QuerySpec` + execution result + ranking (or
failure) to `results/llm_runs/`. Phase 5 computes the actual comparison
metric from what's saved here - this notebook only runs and records.

**Needs `.env` with a real LLM key and a real network connection - this
costs real API calls.** See `paper/PLAN.md`'s open decision (resolved:
single temperature, 0.1) and `paper/comparison_metric.md` for exactly
what gets compared.

**This is the first time this notebook (or this repo's LLM integration)
has run anywhere - more uncertain than phases 1-3.** `LLMQuerySpecGenerator`'s
constructor/`generate()` signature and the env var names were confirmed
from `smart_spatial_system`'s GitHub source (`orchestrator/planning/
llm_spec_generator.py`), but the *exact shape of what the LLM actually
produces* obviously can't be - that's the whole point of this phase.
**Run the single smoke-test cell below first and read its output before
running the full `N`-run loop** - it costs 1 API call instead of 20, and
catches a broken prompt/refs/env setup before you pay for the rest.


## Setup

In [ ]:
import json
import os
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from orchestrator.capability_registry import CapabilityRegistry
from orchestrator.planning.dag_executor import DagExecutor
from orchestrator.planning.planner import DeterministicPlanner
from orchestrator.planning.llm_spec_generator import (
    LLMQuerySpecGenerator,
    LLMSpecGenerationError,
    OpenAICompatibleLLMClient,
    normalize_llm_query_spec_for_planning,
    query_spec_to_dict,
)

load_dotenv("../.env")  # no-op if the file doesn't exist; nothing here needs it except this phase

PROCESSED = Path("../data/processed")
RESULTS = Path("../results")
LLM_RUNS = RESULTS / "llm_runs"
LLM_RUNS.mkdir(parents=True, exist_ok=True)

TARGET_CRS = "EPSG:31256"
# Same exact question text the rule-based arm (phase 3) got - the whole
# comparison rests on "same question, run N times" for both arms. It
# deliberately does NOT state the amenity weights/max_distance_m numbers
# - the LLM has to choose those itself, which is exactly what
# paper/comparison_metric.md's Layer 2 (parametric agreement) measures.
RAW_QUERY = "Rank Vienna's 23 municipal districts by accessibility to metro, schools, and parks"
TEMPERATURE = float(os.environ.get("LLM_TEMPERATURE", "0.1"))
N_RUNS = 20  # paper/PLAN.md's starting point

print("model env var set:", bool(os.environ.get("LLM_MODEL")))
print("an API key is set:", any(os.environ.get(k) for k in ("LLM_API_KEY", "AVALAI_API_KEY", "OPENAI_API_KEY")))
print("temperature:", TEMPERATURE)


## Load phase 2/3 outputs

In [ ]:
def load_geojson(name):
    with open(PROCESSED / f"{name}.geojson") as f:
        return json.load(f)

sites_geojson = load_geojson("districts")
amenity_geojson = {
    "metro": load_geojson("metro"),
    "schools": load_geojson("schools"),
    "parks": load_geojson("parks"),
}
initial_inputs = {"sites": sites_geojson, **amenity_geojson}

rule_based_ranking = pd.read_csv(RESULTS / "rule_based_ranking.csv")
print(f"sites: {len(sites_geojson['features'])}, rule-based reference ranking: {len(rule_based_ranking)} rows")


## LLM client + generator

In [ ]:
llm_client = OpenAICompatibleLLMClient()  # reads LLM_API_KEY/AVALAI_API_KEY/OPENAI_API_KEY, LLM_BASE_URL, LLM_MODEL from env
generator = LLMQuerySpecGenerator(llm_client, temperature=TEMPERATURE)

# system_hints is plain text appended to the generator's own prompt -
# controls exactly what we tell it, unlike the richer (and, for this
# simple GeoJSON-only case, probably unnecessary) `context` dict the
# same method accepts. Tells it the exact ref strings our initial_inputs
# actually uses, without telling it any weight/max_distance numbers, and
# WITHOUT telling it which operations to use - which operations it picks
# and how it wires them up is exactly what this experiment measures.
#
# The hints below grew across THREE rounds of real smoke-test/full-run
# failures - all schema-safety, not plan guidance (they never say which
# operations, weights, or max_distance_m values to pick):
# (1) 'distance_to' missing one of its two required inputs - fixed
# upstream since (see claude/smart-spatial-system-upstream-bugs.md), kept
# as a harmless belt-and-suspenders reminder.
# (2) normalize_llm_query_spec_for_planning()'s real-estate scoring
# default - also fixed upstream, kept for the same reason.
# (3)+(4) NEW (2026-09-13, after a generic "outputs are new, chain them"
# hint alone did NOT fix it): the first full N=20 run (2026-09-12) had
# every one of 20/20 runs wire score_features straight to the ORIGINAL
# pre-distance sites vector - confirmed from each run's own
# __score_details__ (value: None for every factor). Added a generic
# chaining sentence and re-ran (2026-09-13) - EVERY run was STILL
# degenerate, just with a different (also wrong) guessed field name
# ('enriched_distance_to_metro' this time). The generic sentence wasn't
# concrete enough. Fixed by grounding the hint in this repo's OWN
# rule-based arm's actual, confirmed-working query spec
# (results/rule_based_query_spec.json) instead of describing the
# mechanism abstractly: it chains nearest_neighbor calls
# source=sites_metric->output=sites_with_metro, then
# source=sites_with_metro (not the original!)->output=sites_with_schools,
# then source=sites_with_schools->output=sites_with_parks, and ONLY THEN
# feeds sites_with_parks to score_features - and every nearest_neighbor
# call sets an explicit distance_field param naming exactly where the
# result is written. Both batches preserved as-is in
# results/llm_runs/diagnostic_2026-09-12_wiring_bug/ and
# diagnostic_2026-09-13_wiring_bug_round2/ - do not overwrite either;
# they are the documented before-fix evidence (and the round2 batch is
# itself evidence that a too-abstract hint doesn't fix this).
SYSTEM_HINTS = (
    "Available input layers - use these exact ref strings as operation inputs: "
    "'sites' (Point/Polygon layer: the 23 Vienna municipal districts to rank), "
    "'metro' (Point layer: U-Bahn station locations), "
    "'schools' (Point layer: school locations), "
    "'parks' (Point layer: park locations, already reduced to centroids). "
    "Produce one complete, executable QuerySpec: reproject everything to a "
    f"metric CRS ({TARGET_CRS}), compute each site's distance to its nearest "
    "amenity in each of the three layers, combine the three distances into a "
    "single accessibility score, and rank the sites by that score. "
    "Four schema requirements, regardless of which specific operations you "
    "choose: (1) give every operation ALL of the input roles its capability "
    "requires, not just one - for example, an operation that measures "
    "distance between two layers typically needs both a source-layer role "
    "and a separate target-layer role, so check every required role is "
    "supplied. (2) any score_features operation must include a complete "
    "'scoring_spec' with a 'factors' list built from the distance fields "
    "you actually computed (e.g. the distance-to-metro, distance-to-schools, "
    "and distance-to-parks fields) - do not rely on a bare 'score_expression' "
    "string alone, and do not reuse factor or field names from an unrelated "
    "domain (for example real-estate fields like buildable_zone or "
    "flood_risk) that do not exist in this data. (3) each operation's "
    "'output' name refers to a NEW result; it does NOT mutate the vector(s) "
    "it was given in place, so a later operation only sees fields that "
    "are actually present on whichever specific output name you give it. "
    "To combine distances computed against more than one amenity layer "
    "onto a single vector before scoring, you must CHAIN the computations: "
    "compute the distance to the first amenity layer from the reprojected "
    "sites vector, producing a new enriched result; then compute the "
    "distance to the SECOND amenity layer using THAT enriched result as "
    "the source/vector input (not the original sites vector again), "
    "producing a further-enriched result; repeat for every remaining "
    "amenity layer. Only the final, fully-chained result contains every "
    "distance field together - that final chained result, not any of the "
    "original per-layer results, is what score_features must take as its "
    "input. (4) whichever operation you use to compute a distance, set "
    "its parameter that names the output field explicitly (for example a "
    "'distance_field' parameter) to a specific, unique field name for "
    "each computation - do not leave that parameter unset or empty and do "
    "not assume a default field name - and reference that exact same "
    "field name later when building score_features's factors."
)
print(SYSTEM_HINTS)


## Smoke test - ONE run first

Costs 1 API call. Read the printed operation sequence and (if it
executes) the ranking before running the full loop below - if the refs
are wrong or the plan doesn't execute, fix `SYSTEM_HINTS` here rather
than after spending on `N_RUNS`.


In [ ]:
def generate_one(raw_query=RAW_QUERY, system_hints=SYSTEM_HINTS):
    """One LLM call -> normalized QuerySpec. Raises LLMSpecGenerationError on
    a malformed/unparseable LLM response - let that propagate here (smoke
    test), catch it per-run in the full loop below."""
    spec = generator.generate(raw_query, system_hints=system_hints)
    # Defensive: apply the package's own normalization explicitly rather
    # than assume generate() already did it internally - idempotent if it
    # did, and safer if it didn't.
    return normalize_llm_query_spec_for_planning(spec)

test_spec = generate_one()
test_op_sequence = [op.name if hasattr(op, "name") else str(op) for op in test_spec.operations]
print(f"{len(test_op_sequence)} operations:")
for name in test_op_sequence:
    print(" ", name)


In [ ]:
registry = CapabilityRegistry.from_plugin_modules(tolerant=True)

def run_pipeline(query_spec):
    """Compile + execute a QuerySpec. Same generic DeterministicPlanner /
    DagExecutor pipeline phase 3 used - it doesn't care whether the spec
    came from the rule-based builder or the LLM."""
    plan = DeterministicPlanner().build(query_spec)
    return DagExecutor(lambda name: registry.resolve(name).callable).execute(
        plan, initial_inputs=initial_inputs,
    )

def vector_out_to_dataframe(vector_out):
    """VectorOut (geochat_sdk.types.vector.VectorOut) only has two real
    attributes: `.features` (a list of GeoJSON-style Feature dicts, each
    with a "properties" key) and `.metadata` - confirmed from source,
    including smart_spatial_system's own plugins/report_builder.py, which
    does this exact extraction internally to build a ReportOut. There is
    no `.to_dataframe()`/`.to_dict()`/`.records` - this is the correct,
    general way to read one, not a guess."""
    features = getattr(vector_out, "features", None)
    if features is None:
        raise TypeError(f"expected a VectorOut-like object with .features, got {type(vector_out)}")
    return pd.DataFrame([f.get("properties", {}) if isinstance(f, dict) else {} for f in features])

def plan_score_and_rank_fields(query_spec, default_score_field="accessibility_score", default_rank_field="rank"):
    """Read the actual score_field/rank_field names the plan itself
    declares (typically on its rank_features op, sometimes build_report),
    instead of assuming our own default names - the LLM is free to call
    the score column "score", "accessibility_score", or anything else,"""
    score_field, rank_field = default_score_field, default_rank_field
    for op in query_spec.operations:
        params = getattr(op, "params", {}) or {}
        if "score_field" in params:
            score_field = params["score_field"]
        if "rank_field" in params:
            rank_field = params["rank_field"]
    return score_field, rank_field

def extract_ranking(result, query_spec):
    """Confirmed in phase 3: a bare VectorOut (e.g. outputs["sites_ranked"])
    can't go through pd.DataFrame() directly. A `build_report` step (giving
    outputs["sites_report"], a ReportOut with a plain .table["rows"]) fixes
    that - but it turns out to be OPTIONAL: smart_spatial_system only adds
    it when the LLM's own spec.outputs declares a report/pdf output
    (confirmed from source, orchestrator/planning/llm_spec_generator.py's
    _should_inject_report()) - the LLM choosing not to is a legitimate plan,
    not a bug, so this has to handle both cases:
      - outputs["sites_report"] present -> use its .table["rows"] (phase 3's
        confirmed path).
      - otherwise -> read the plan's own LAST operation's `output=` name
        (nothing later in the plan consumes it, so it's the terminal
        result) straight out of `result.outputs`, and convert that
        VectorOut via vector_out_to_dataframe() above.
    Either way, the score/rank column names come from the plan's own
    declared score_field/rank_field (plan_score_and_rank_fields above), with
    a same fallback to an unambiguous "score"/"*_score" column only if
    the declared name is genuinely missing (e.g. renamed by a package
    quirk) - not a first resort."""
    report = result.outputs.get("sites_report")
    if report is not None and hasattr(report, "table"):
        df = pd.DataFrame(report.table["rows"])
    else:
        terminal_name = query_spec.operations[-1].output
        terminal = result.outputs.get(terminal_name)
        if terminal is None:
            raise KeyError(
                f"neither 'sites_report' nor the plan's terminal output {terminal_name!r} "
                f"found in result.outputs (keys: {list(result.outputs.keys())})"
            )
        df = vector_out_to_dataframe(terminal)

    score_field, rank_field = plan_score_and_rank_fields(query_spec)
    if rank_field not in df.columns:
        raise KeyError(f"expected rank column {rank_field!r} not in columns {list(df.columns)}")
    if score_field not in df.columns:
        score_like = [c for c in df.columns if c == "score" or (c.endswith("_score") and c != rank_field)]
        if len(score_like) == 1:
            df = df.rename(columns={score_like[0]: score_field})
        else:
            raise KeyError(
                f"expected a score column (tried {score_field!r}, found no unambiguous "
                f"fallback among {list(df.columns)})"
            )
    return df.sort_values(rank_field).reset_index(drop=True)

def ranking_is_degenerate(df, score_field):
    """Cheap sanity check for the exact failure mode found in the first
    full N=20 run (2026-09-12, preserved in
    results/llm_runs/diagnostic_2026-09-12_wiring_bug/ - do not overwrite
    that folder): score_features wired to the pre-distance vector instead
    of any distance_to output, so every score came out identical (0.0) and
    rank collapsed to input order. A real plan CAN legitimately tie many
    sites (phase 3's rule-based arm ties 21/23 at 100.0), so "all scores
    equal" alone isn't proof of a bug - but it's exactly the signature to
    flag and look at, not silently accept."""
    return bool(df[score_field].nunique(dropna=False) <= 1)

test_result = run_pipeline(test_spec)
print("execution success:", test_result.success)
if test_result.success:
    test_score_field, _ = plan_score_and_rank_fields(test_spec)
    test_ranking = extract_ranking(test_result, test_spec)
    display(test_ranking)
    if ranking_is_degenerate(test_ranking, test_score_field):
        print(
            "WARNING: every score is identical - this is the exact signature of the "
            "2026-09-12 wiring bug (score_features never actually fed by any "
            "distance_to output). Check the operation sequence printed above: does "
            "score_features's 'vector' input come from a distance-computing "
            "operation (directly or via a merge), not straight from crs_transform's "
            "output? If so, fix SYSTEM_HINTS and re-run this smoke test before "
            "spending on the full loop."
        )
else:
    print("error:", getattr(test_result, "error", None) or getattr(test_result, "structured_error", None))


**Stop here and check before continuing.** Does the operation sequence
look like a real accessibility pipeline (reproject, nearest-neighbor per
amenity, score, rank)? Did it execute? Does the ranking look plausible
(not all-NaN, not identical to the rule-based reference by suspicious
coincidence)? If not, fix `SYSTEM_HINTS` above and re-run the smoke test
- don't run the full loop on a broken prompt.


## Full N-run loop

Each run: generate a fresh QuerySpec (no caching/reuse - a new LLM call every time, matching "same question, run N times"), try to execute it, save everything regardless of success. A generation or execution failure is itself data (`paper/comparison_metric.md`'s "Success rate" reliability metric) - it is recorded, not retried.

In [ ]:
run_records = []

for i in range(N_RUNS):
    t0 = time.monotonic()
    record = {
        "run_index": i, "generation_success": False, "execution_success": False,
        "degenerate_ranking": None, "error": None,
    }

    try:
        spec = generate_one()
        record["generation_success"] = True
        record["query_spec"] = query_spec_to_dict(spec)
        record["operation_sequence"] = [op.name if hasattr(op, "name") else str(op) for op in spec.operations]
    except LLMSpecGenerationError as e:
        record["error"] = f"generation: {e}"
        record["latency_s"] = time.monotonic() - t0
        run_records.append(record)
        print(f"run {i}: generation FAILED - {e}")
        continue

    try:
        result = run_pipeline(spec)
        record["execution_success"] = bool(result.success)
        if result.success:
            ranking_df = extract_ranking(result, spec)
            record["ranking"] = ranking_df.to_dict(orient="records")
            run_score_field, _ = plan_score_and_rank_fields(spec)
            # Flags, doesn't discard - the 2026-09-12 wiring bug produced
            # exactly this signature (execution_success=True, every score
            # identical) in 20/20 runs. Kept as a separate field rather
            # than folded into execution_success, since the plan DID run
            # without error - this is a semantic check the DAG executor
            # itself doesn't make, not a crash.
            record["degenerate_ranking"] = ranking_is_degenerate(ranking_df, run_score_field)
        else:
            record["error"] = f"execution: {getattr(result, 'error', None) or getattr(result, 'structured_error', None)}"
    except Exception as e:  # noqa: BLE001 - a bad LLM-produced spec can fail in many ways; record, don't crash the loop
        record["error"] = f"execution: {type(e).__name__}: {e}"

    record["latency_s"] = time.monotonic() - t0
    run_records.append(record)

    (LLM_RUNS / f"run_{i:02d}.json").write_text(json.dumps(record, indent=2))
    print(f"run {i}: generation={record['generation_success']} execution={record['execution_success']} "
          f"degenerate={record['degenerate_ranking']} latency={record['latency_s']:.1f}s")


## Save the run manifest

A lightweight per-run summary - phase 5 still reads the full `run_{i:02d}.json` files for the actual PAR/parametric/rank-stability computation, this is just an index.

In [ ]:
manifest = pd.DataFrame([
    {
        "run_index": r["run_index"],
        "generation_success": r["generation_success"],
        "execution_success": r["execution_success"],
        "degenerate_ranking": r.get("degenerate_ranking"),
        "latency_s": r["latency_s"],
        "n_operations": len(r.get("operation_sequence", [])),
        "error": r["error"],
    }
    for r in run_records
])
manifest.to_csv(LLM_RUNS / "manifest.csv", index=False)

success_rate = manifest["execution_success"].mean()
print(f"success rate: {success_rate:.1%} ({manifest['execution_success'].sum()}/{len(manifest)})")
print(f"latency (s): median={manifest['latency_s'].median():.1f}  "
      f"IQR=[{manifest['latency_s'].quantile(0.25):.1f}, {manifest['latency_s'].quantile(0.75):.1f}]")
n_degenerate = manifest["degenerate_ranking"].sum()
if n_degenerate:
    print(
        f"WARNING: {int(n_degenerate)}/{manifest['execution_success'].sum()} successful runs have a "
        "DEGENERATE ranking (every score identical - the 2026-09-12 wiring-bug signature). "
        "Do not treat these as valid rankings for phase 5 without checking their query_spec first."
    )
manifest


## Final checks

In [ ]:
check = pd.read_csv(LLM_RUNS / "manifest.csv")
assert len(check) == N_RUNS, f"expected {N_RUNS} run records, found {len(check)}"

n_saved_files = len(list(LLM_RUNS.glob("run_*.json")))
assert n_saved_files == N_RUNS, f"expected {N_RUNS} run_*.json files, found {n_saved_files}"

if check["execution_success"].sum() == 0:
    print("WARNING: every run failed to execute - don't proceed to phase 5 until at least some runs succeed. "
          "Check results/llm_runs/run_00.json's 'error' field first.")
elif check["degenerate_ranking"].sum() == check["execution_success"].sum():
    print(
        "WARNING: every successful run has a DEGENERATE ranking (every score identical - "
        "the 2026-09-12 wiring-bug signature). Executing without error is not the same as "
        "producing a meaningful ranking - don't proceed to phase 5 on this data. Check a "
        "run's query_spec: does score_features's 'vector' input actually come from a "
        "distance-computing operation?"
    )
else:
    n_ok = int(check["execution_success"].sum() - check["degenerate_ranking"].sum())
    print(f"all checks passed - {check['execution_success'].sum()}/{N_RUNS} runs executed successfully "
          f"({n_ok} with a non-degenerate ranking)")

check


## Next: phase 5

`notebooks/04_comparison_metric.ipynb` reads `results/llm_runs/run_*.json`
(this phase) and `results/rule_based_ranking.csv` /
`results/rule_based_query_spec.json` (phase 3), and computes the actual
three-layer metric from `paper/comparison_metric.md`:

- **Layer 1 (PAR):** mode `operation_sequence` across the successful
  `run_*.json` files, agreement rate against it.
- **Layer 2 (parametric):** this phase's `query_spec` field has each
  run's chosen operation parameters (weights/`max_distance_m` if the LLM
  surfaced them in a `score_features`-equivalent step) - mean/sd per
  amenity, among runs sharing the reference sequence.
- **Layer 3 (Rank Stability):** pairwise Spearman rho over each run's
  `ranking` field (by district `name`, same join key phase 3 used), plus
  rho against `results/rule_based_ranking.csv` as the fixed reference.
- **Reliability:** success rate and latency are already in
  `results/llm_runs/manifest.csv` - reuse it, don't recompute.
